In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report,confusion_matrix,f1_score,make_scorer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder,StandardScaler
from imblearn.over_sampling import SMOTE

In [57]:
le=LabelEncoder()
scaler=StandardScaler()
model_rfc=RandomForestClassifier(random_state=42,n_estimators=100)
model_lreg=LogisticRegression(random_state=42)
smote=SMOTE(random_state=42)

In [27]:
df=pd.read_csv('../data/cancer-risk-factors.csv')

In [28]:
df.head(5)

,Patient_ID,Cancer_Type,Age,Gender,Smoking,Alcohol_Use,Obesity,Family_History,Diet_Red_Meat,Diet_Salted_Processed,...,Physical_Activity,Air_Pollution,Occupational_Hazards,BRCA_Mutation,H_Pylori_Infection,Calcium_Intake,Overall_Risk_Score,BMI,Physical_Activity_Level,Risk_Level
0,LU0000,Breast,68,0,7,2,8,0,5,3,...,4,6,3,1,0,0,0.398696,28.0,5,Medium
1,LU0001,Prostate,74,1,8,9,8,0,0,3,...,1,3,3,0,0,5,0.424299,25.4,9,Medium
2,LU0002,Skin,55,1,7,10,7,0,3,3,...,1,8,10,0,0,6,0.605082,28.6,2,Medium
3,LU0003,Colon,61,0,6,2,2,0,6,2,...,6,4,8,0,0,8,0.318449,32.1,7,Low
4,LU0004,Lung,67,1,10,7,4,0,6,3,...,9,10,9,0,0,5,0.524358,25.1,2,Medium


In [ ]:
x=df.drop(columns=['Patient_ID','Cancer_Type','Risk_Level'])
y=le.fit_transform(df['Risk_Level'])

In [30]:
x

,Age,Gender,Smoking,Alcohol_Use,Obesity,Family_History,Diet_Red_Meat,Diet_Salted_Processed,Fruit_Veg_Intake,Physical_Activity,Air_Pollution,Occupational_Hazards,BRCA_Mutation,H_Pylori_Infection,Calcium_Intake,Overall_Risk_Score,BMI,Physical_Activity_Level
0,68,0,7,2,8,0,5,3,7,4,6,3,1,0,0,0.398696,28.0,5
1,74,1,8,9,8,0,0,3,7,1,3,3,0,0,5,0.424299,25.4,9
2,55,1,7,10,7,0,3,3,4,1,8,10,0,0,6,0.605082,28.6,2
3,61,0,6,2,2,0,6,2,4,6,4,8,0,0,8,0.318449,32.1,7
4,67,1,10,7,4,0,6,3,10,9,10,9,0,0,5,0.524358,25.1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,60,1,4,6,4,0,10,6,4,4,5,3,1,0,4,0.437539,30.3,3
1996,84,1,5,7,8,0,10,0,1,2,1,3,0,0,2,0.451128,25.9,4
1997,65,0,7,2,10,0,4,2,2,3,6,0,0,1,0,0.295760,22.5,3
1998,64,1,10,2,10,0,2,10,7,5,4,2,0,0,10,0.422201,25.3,3


In [31]:
y

array([2, 2, 2, ..., 1, 2, 2], shape=(2000,))

## why cancer_type was removed from the features list

- It's not a predictive input - it's an outcome label or categorical grouping that's already strongly correlated with Risk Level

- If we include it, the model will cheat by learning the mapping like Prostate --> High risk, instead of learning from true risk factors (smoking, BMI etc)

In [32]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [33]:
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

### Randomfoest Classifier

In [34]:
model_rfc.fit(x_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [48]:
model_rfc_y_pred=model_rfc.predict(x_test)

print("Classification Report :")
print(classification_report(y_test,model_rfc_y_pred))

print("confusion Matrix :")
print(confusion_matrix(y_test,model_rfc_y_pred))


Classification Report :
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        20
           1       1.00      1.00      1.00        65
           2       1.00      1.00      1.00       315

    accuracy                           1.00       400
   macro avg       1.00      0.98      0.99       400
weighted avg       1.00      1.00      1.00       400

confusion Matrix :
[[ 19   0   1]
 [  0  65   0]
 [  0   0 315]]


The model accurately distinguishes all risk levels, with only 1 mistake out of 400 with imbalanced dataset.

### Logistic Regression

In [36]:
model_lreg.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [47]:
model_lreg_y_pred=model_lreg.predict(x_test)


print("Classification Report :")
print(classification_report(y_test,model_lreg_y_pred))

print("Confusion Matrix :")
print(confusion_matrix(y_test,model_lreg_y_pred))

Classification Report :
              precision    recall  f1-score   support

           0       1.00      0.80      0.89        20
           1       0.97      0.95      0.96        65
           2       0.98      0.99      0.99       315

    accuracy                           0.98       400
   macro avg       0.98      0.92      0.95       400
weighted avg       0.98      0.98      0.98       400

Confusion Matrix :
[[ 16   0   4]
 [  0  62   3]
 [  0   2 313]]


### Removing Overall_Risk_Score and retrying

In [38]:
x_new=df.drop(columns=['Patient_ID','Cancer_Type','Risk_Level','Overall_Risk_Score'])
le_new=LabelEncoder()
y_new=le_new.fit_transform(df['Risk_Level'])

In [39]:
x_train_new,x_test_new,y_train_new,y_test_new=train_test_split(x_new,y_new,test_size=0.2,random_state=42,stratify=y)

In [40]:
scaler_new=StandardScaler()
x_train_new=scaler_new.fit_transform(x_train_new)
x_test_new=scaler_new.transform(x_test_new)

In [41]:
model_rfc_new=RandomForestClassifier(random_state=42,n_estimators=100)
model_rfc_new.fit(x_train_new,y_train_new)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [46]:
model_rfc_y_pred_new=model_rfc_new.predict(x_test_new)


print("classification report : ")
print(classification_report(y_test_new,model_rfc_y_pred_new,target_names=le_new.classes_))

print("Confusion Matrix :")
print(confusion_matrix(y_test_new,model_rfc_y_pred_new))

classification report : 
              precision    recall  f1-score   support

        High       1.00      0.05      0.10        20
         Low       0.85      0.34      0.48        65
      Medium       0.83      0.99      0.90       315

    accuracy                           0.83       400
   macro avg       0.89      0.46      0.49       400
weighted avg       0.84      0.83      0.80       400

Confusion Matrix :
[[  1   0  19]
 [  0  22  43]
 [  0   4 311]]


In [52]:
pd.DataFrame(df.Risk_Level.value_counts())

,count
Risk_Level,
Medium,1574
Low,324
High,102


### SMOTE - Synthetic Minority Over-Sampling Technique

In [53]:
x=df.drop(columns=['Patient_ID','Cancer_Type','Risk_Level','Overall_Risk_Score'])
y=df['Risk_Level']

In [54]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [60]:
x_train_res,y_train_res=smote.fit_resample(x_train,y_train)


print("Class Distribution :")
print(y_train_res.value_counts())

Class Distribution :
Risk_Level
Medium    1259
Low       1259
High      1259
Name: count, dtype: int64


In [61]:
model_rfc_smote=RandomForestClassifier(random_state=42)
model_rfc_smote.fit(x_train_res,y_train_res)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [65]:
model_rfc_smote_y_pred=model_rfc_smote.predict(x_test)


print("Classification Report : ")
print(classification_report(y_test,model_rfc_smote_y_pred))

print("Confusion Matrix :")
print(confusion_matrix(y_test,model_rfc_smote_y_pred))

Classification Report : 
              precision    recall  f1-score   support

        High       0.41      0.35      0.38        20
         Low       0.73      0.58      0.65        65
      Medium       0.88      0.92      0.90       315

    accuracy                           0.84       400
   macro avg       0.67      0.62      0.64       400
weighted avg       0.83      0.84      0.83       400

Confusion Matrix :
[[  7   0  13]
 [  0  38  27]
 [ 10  14 291]]


- Accuracy: 84%
- High Risk: 7 correctly classified, 13 mis-classified as Medium
- Low Risk: 38 correctly classified, 27 confused with Medium
- Medium Risk: 291 correctly classified, 24 confused as High/Low

- Still struggles to detect High-risk patients even with SMOTE

## Optuna